# Calculating Power-law scaling in fluctuations of information

In [1]:
import gc
import os
import numpy as np
import pandas as pd

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
from scipy.stats import t

import warnings;warnings.filterwarnings('ignore')

In [2]:
CORPUS_NAME = 'null'

In [3]:
DATA_PATH = '../../../updated-data/obs/lme-ready'
NULL_DATA_PATH = '../../../updated-data/null/null-lme-ready'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [4]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [5]:
fs = [
    f for f in os.listdir(NULL_DATA_PATH)
    if ('xling-' not in f)
    and (not f.startswith('.'))
    and (f.endswith('.parquet'))
]
len(fs)

4522

In [6]:
fs[:3]

['CABNC-CABNC-KBWRE01F-KBW.parquet',
 'CABNC-CABNC-KB8RE00L-KB8.parquet',
 'CABNC-CABNC-KC4RE00C-KC4.parquet']

### Function to calculate AVAR

In [7]:
# def add_avar_col(df, val_col, merge_cols):
#     # add next step in series
#     df[val_col + '_2'] = None
#     df[val_col + '_3'] = None
#
#     groups = df.groupby(by=merge_cols)
#     completed = []
#     try:
#         for _, dfi in groups:
#             idx = dfi.index
#             dfi[val_col + '_2'].loc[idx[:-1]] = dfi[val_col].values[1:]
#             dfi[val_col + '_3'].loc[idx[:-2]] = dfi[val_col].values[2:]
#
#             dfi['AVAR_'] = (1 / (2 * (len(dfi) ** 2))) * (dfi[val_col + '_3'] - (2 * dfi[val_col + '_2']) + dfi[val_col]) ** 2
#             completed+=[dfi]
#         df_ = pd.concat(completed, ignore_index=True)
#
#         del df_[val_col + '_2']
#         del df_[val_col + '_3']
#
#     except Exception:
#         df_ = df
#
#     gc.collect()
#
#     return df_

def add_avar_col(df, val_col, merge_cols):
    # add next step in series
    df[val_col + '_2'] = None
    df[val_col + '_3'] = None

    groups = df.groupby(by=merge_cols)
    completed = []
    for _, dfi in groups:
        idx = dfi.index
        dfi[val_col + '_2'].loc[idx[:-1]] = dfi[val_col].values[1:]
        dfi[val_col + '_3'].loc[idx[:-2]] = dfi[val_col].values[2:]
        dfi['AVAR'] = (1 / (2 * (len(dfi) ** 2))) * (dfi[val_col + '_3'] - (2 * dfi[val_col + '_2']) + dfi[val_col]) ** 2
        completed+=[dfi]
    df = pd.concat(completed, ignore_index=True)

    del df[val_col + '_2']
    del df[val_col + '_3']
    gc.collect()

    return df

def lang_id(x):
    if 'xling-' in x:
        return x.split('-')[2]
    else:
        return 'eng'

def corpus_id(x):
    if 'xling-' in x:
        return x.split('-')[1]
    else:
        return x.split('-')[0]

## Processing individual datasets

In [8]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [9]:
COEF_VAR = 'AVAR'

In [10]:
param_list = 'time_delta'

In [20]:
merge_cols = ['x_turn_id', 'self']
# merge_cols = ['x_turn_id', 'y_speaker']

In [21]:
df_scale, df_regression, k_docs, pct, error_tracker = [], [], dict(), 1., 0

In [22]:
for f in tqdm(fs):

    table = pq.ParquetFile(os.path.join(NULL_DATA_PATH, f))

    df = table.read(columns=[
        'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
        # COEF_VAR,
        'x_turn_id', 'y_turn_id', 'I'
        # 'nx', 'ny'
    ]).to_pandas()

    df['self'] = (df['x_speaker'] == df['y_speaker'])
    # df['self'] = False
    # df['y_speaker'].loc[~df['self']] = df['y_speaker'].loc[~df['self']].sample(frac=1.)

    # df = df.sample(frac=1.).sample(frac=1.)
    # df.index = range(len(df))

    df = add_avar_col(df, 'I', merge_cols)

    df = df.loc[~df[COEF_VAR].isna()]

    groups = df.groupby(by=['x_speaker', 'self'])
    for labels, dfi in groups:
        self_other = dfi['self'].unique()[0]
        dfi['min_y_turn_id'] = dfi.groupby('x_turn_id')['y_turn_id'].transform('min')

        try:
            if len(dfi) > 1:
                # print(len(dfi))
                # dfi['I'] = dfi['I'].sample(frac=1.)

                # dfi = add_avar_col(dfi, 'I', ['self', 'x_turn_id'])

                # dfi['AVAR_'] += 1e-5

                # k_docs[CORPUS_NAME][labels[2]] += 1

                if dfi[COEF_VAR].isna().mean() < 1:

                    b, a = np.polyfit(
                        np.log((dfi['y_turn_id'] - dfi['min_y_turn_id']).astype(float).sample(frac=1.).values + 1),
                        np.log(dfi[COEF_VAR].astype(float).values),
                        1
                    )

                    df_regression += [
                        {
                            'corpus': corpus_id(f),
                            'cid': f.replace('.parquet', ''),
                            'speaker': labels[0],
                            'self': self_other,
                            'a': np.exp(a),
                            'b': b,
                            'k': len(dfi)
                        }
                    ]

        except Exception as exx:
            # print(exx)
            error_tracker += 1
            err_df = (df, dfi)

df_regression = pd.DataFrame(df_regression)

  0%|          | 0/4522 [00:00<?, ?it/s]

** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  5 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  5 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number 

In [23]:
len(df_regression), error_tracker

(16567, 700)

In [16]:
# dfi.head()

In [ ]:
# df_regression.isna().sum()

In [ ]:
# df_regression = df_regression.loc[
#     (df_regression.isna().astype(int).sum(axis=1) == 0)
#     & (np.isinf(df_regression).astype(int).sum(axis=1) == 0)
# ]

In [29]:
df_regression.to_parquet(
    os.path.join(OUTPUTS_PATH,'null-all.parquet'),
    compression='snappy',
    engine='fastparquet'
)

## Aggregate statistics

In [24]:
split_by = ['self']

In [25]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values

df0['k'] = df_regression[split_by+['b']].groupby(by=split_by).agg('count').reset_index()['b'].values

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [26]:
df0['t'] = df0['b'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat,k in df0[['t', 'k']].values]

In [27]:
df0.head()

,self,b,std,k,se,t,p
0,False,-0.001426,0.336332,7538,0.003874,-0.36817,0.356378
1,True,-0.007200,0.364725,5351,0.004986,-1.44406,0.074390


In [28]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'null-results.csv'),
    index=False, encoding='utf-8'
)

##### Deep aggregate...

In [ ]:
b, std, k = df_regression['b'].mean(), df_regression['b'].std(), len(df_regression)
tstat = b / (std / np.sqrt(k))
b, std, tstat, t.sf(tstat.__abs__(), df=k-1)

## Corpus level analysis of results

##### If you have prior data ...

In [ ]:
df_regression = pd.read_parquet(os.path.join(OUTPUTS_PATH,'null-all.parquet'))

##### otherwise

In [ ]:
split_by = ['corpus', 'self']

In [ ]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values

df0['k'] = df_regression[split_by+['b']].groupby(by=split_by).agg('count').reset_index()['b'].values

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [ ]:
df0['t'] = df0['b'] / df0['se']

In [ ]:
df0.head(n=20)

In [ ]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, CORPUS_NAME+'.csv'),
    index=False, encoding='utf-8'
)